# L4 Vertebra Segmentation, Bone Mineral Density & Microarchitecture — *Production Pipeline v2*

### *"The measurement of spinal bone architecture following hip surgery using X-ray datasets"*

This notebook implements a **fully self-contained PyTorch U-Net pipeline** that segments the
lumbar vertebrae **L1–L5** from 2-D antero-posterior (AP) X-rays, isolates the **L4** vertebral
body, and quantifies two downstream clinical biomarkers:

1. **Bone Mineral Density (BMD)** — an area-based density proxy from normalized grayscale intensity.
2. **Bone Microarchitecture (Tissue Quality)** — 2-D texture descriptors from the
   Gray-Level Co-occurrence Matrix (GLCM).

Unlike the YOLOv8-seg prototype (`L4_AP_segmentation.ipynb`), this version trains a
**ResNet-34 U-Net (ImageNet-pretrained encoder)** with a hybrid **Dice + BCE** objective, uses a
**robust positional L4 selector** so the target vertebra is essentially never dropped, and adds an
**Explainable-AI (Grad-CAM)** report so the segmentation decision can be clinically audited.

**Pipeline (7 steps + XAI):**
`Data Loading → Preprocessing → Train/Val/Test Split → Model Building → Training →
Evaluation (BMD + GLCM) → Pre-trained Model & Inference → Grad-CAM XAI`

> ⚠️ **Clinical caveat.** Planar-radiograph "density" is an *uncorrected radiological proxy*
> (pixel intensity), **not** true areal BMD (g/cm²). True BMD requires DXA/QCT or an in-image
> calibration phantom (e.g. an aluminium step-wedge). All values below are relative indices
> intended for methodological demonstration.

## What changed vs. v1 — fixing frequent L4 extraction failures

The first U-Net revision dropped the L4 mask too often. Root causes and fixes:

| Issue (previous v2) | Why it failed | Fix in this notebook |
|---|---|---|
| **U-Net trained from scratch**, 256 px | Weak features; poor JPG→DICOM generalization | **ResNet-34 encoder pretrained on ImageNet**, 320 px |
| **L4 read from channel-3 only**, threshold 0.5 | If the channel is under-confident the mask is **empty** | **Robust positional selector**: order vertebrae top→bottom, take the **4th** (= L4), with channel cross-check |
| No fallback on failure | One weak prediction ⇒ no L4 | Multi-stage fallback (threshold relaxation) so a mask is **always** returned |
| Flip-only augmentation | Limited pose variation | Added light **rotation/scale** augmentation |

This mirrors the YOLOv8 prototype's resilience (which always recovered L4 via a *4th-from-top*
fallback) while keeping a clean, dependency-light PyTorch implementation.

## 1. Clinical Definitions & Mathematical Formulation

### 1.1 Bone Mineral Density (BMD) — area-based intensity proxy

After min–max normalization of the X-ray to $[0,1]$, the L4 region of interest (ROI) is the set of
pixels $\Omega_{L4}$ selected by the predicted segmentation mask. The BMD index is the mean
normalized intensity inside the ROI:

$$
\text{BMD}_{\text{index}} \;=\; \frac{1}{|\Omega_{L4}|}\sum_{(x,y)\in\Omega_{L4}} I_{\text{norm}}(x,y),
\qquad I_{\text{norm}}(x,y)=\frac{I(x,y)-I_{\min}}{I_{\max}-I_{\min}}.
$$

Higher mean intensity ⇒ greater radiographic attenuation ⇒ higher apparent bone density.
We additionally report the **median**, **standard deviation** and the **10th/90th percentiles**
to characterise the intensity distribution robustly against endplate/cortex outliers.

### 1.2 Bone Microarchitecture — Gray-Level Co-occurrence Matrix (GLCM)

Trabecular complexity is captured by the **GLCM** $P_{\delta}$, where $P_{\delta}(i,j)$ counts how
often a pixel of gray level $i$ is separated by an offset $\delta=(d,\theta)$ from a pixel of level
$j$. We quantise the ROI to $L=32$ gray levels and average the matrix over
$\theta\in\{0^\circ,45^\circ,90^\circ,135^\circ\}$ at distance $d=1$ to obtain rotation-robust
descriptors (Haralick features):

$$
\textbf{Contrast}=\sum_{i,j}(i-j)^2\,P(i,j), \qquad
\textbf{Energy/ASM}=\sum_{i,j}P(i,j)^2,
$$

$$
\textbf{Correlation}=\sum_{i,j}\frac{(i-\mu_i)(j-\mu_j)\,P(i,j)}{\sigma_i\,\sigma_j}, \qquad
\textbf{Homogeneity}=\sum_{i,j}\frac{P(i,j)}{1+|i-j|}.
$$

**Clinical reading:** healthy dense trabecular bone yields *high Energy/Homogeneity* and
*low Contrast* (uniform texture); osteoporotic/disrupted bone shows *higher Contrast* and
*lower Correlation* as the trabecular network fragments.

## 0. Environment, Installation & Imports

Target environment: **Windows + RTX 5060 Laptop GPU (CUDA)**. Uncomment the `pip` line on first
run. `torch`/`torchvision` must match your CUDA toolkit.

In [ ]:
# !pip install torch torchvision pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg \
# !             opencv-python-headless scikit-image scikit-learn matplotlib seaborn pandas tqdm

import os, glob, re, random, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from skimage.feature import graycomatrix, graycoprops
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device :", DEVICE, "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))

### Global configuration & paths

In [ ]:
# -------------------- Paths --------------------
BUU_ROOT     = "./BUU-LSPINE-2000/AP"
IMG_DIR      = os.path.join(BUU_ROOT, "images")
LBL_DIR      = os.path.join(BUU_ROOT, "labels")
TEST_DCM_DIR  = "./dataset-dcm/test/gather"                          # DICOM test images
TEST_MASK_DIR = "./dataset-dcm/test/gather_mask/SegmentationClass"   # L4 ground-truth masks (PNG)
OUT_DIR       = "./l4_v2_output"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------- Task definition --------------------
CLASS_NAMES = ["L1", "L2", "L3", "L4", "L5"]   # 5 segmentation channels (multi-mask)
N_CLASSES   = len(CLASS_NAMES)
L4_IDX      = CLASS_NAMES.index("L4")          # = 3  (the target vertebra)

# -------------------- Label geometry (BUU-LSPINE) --------------------
LINES_PER_VERTEBRA = 2      # upper + lower endplate lines
N_VERTEBRAE        = 5

# -------------------- Image / training hyper-parameters --------------------
IMG_SIZE    = 320          # square input (must be divisible by 32 for the ResNet encoder)
VAL_RATIO   = 0.20         # 20% held out from BUU-LSPINE for validation
BATCH_SIZE  = 8            # RTX 5060 laptop @ 320 + ResNet34; drop to 4 if you hit CUDA OOM
EPOCHS      = 40           # pretrained encoder converges fast; raise for a final run
LR          = 1e-3
GLCM_LEVELS = 32           # gray-level quantisation for texture
ROI_SHRINK  = 0.85         # shrink mask toward centre to sample cancellous bone

BEST_MODEL_PATH = os.path.join(OUT_DIR, "best_model.pth")
print(f"Target vertebra: {CLASS_NAMES[L4_IDX]} (channel {L4_IDX}) | input {IMG_SIZE}x{IMG_SIZE}")

### Label → polygon → multi-mask utilities

Each BUU-LSPINE `.csv` label holds **10 lines** (`xL,yL,xR,yR,cls`): two endplate lines per
vertebra. We convert the 4 corner points of each vertebra into an ordered quadrilateral
(`TL→TR→BR→BL`), sort the 5 vertebrae top→bottom to assign **L1…L5**, and rasterise them into a
`(5, H, W)` binary mask stack — one channel per vertebra.

In [ ]:
def _rows(path):
    out = []
    for line in open(path):
        nums = re.findall(r"-?\d+\.?\d*", line)
        if len(nums) >= 4:
            out.append([float(v) for v in nums[:4]])   # xL, yL, xR, yR
    return out

def order_quad(pts):
    '''4 points -> [TL, TR, BR, BL] (clockwise, non-self-intersecting).'''
    pts = sorted(pts, key=lambda p: p[1])      # by y
    top = sorted(pts[:2], key=lambda p: p[0])  # upper two: left, right
    bot = sorted(pts[2:], key=lambda p: p[0])  # lower two: left, right
    return [top[0], top[1], bot[1], bot[0]]

def parse_polys(path):
    '''Return list of 5 quads ordered L1..L5 (top->bottom), or [] if malformed.'''
    rows = _rows(path)
    if len(rows) < LINES_PER_VERTEBRA * N_VERTEBRAE:
        return []
    verts = []
    for i in range(N_VERTEBRAE):
        pts = []
        for j in range(LINES_PER_VERTEBRA):
            xL, yL, xR, yR = rows[i * LINES_PER_VERTEBRA + j]
            pts += [(xL, yL), (xR, yR)]
        quad = order_quad(pts)
        cy = sum(p[1] for p in quad) / 4.0
        verts.append((cy, quad))
    verts.sort(key=lambda v: v[0])             # top -> bottom => L1..L5
    return [q for _, q in verts]

def polys_to_maskstack(polys, H, W, out_size, shrink=1.0):
    '''Rasterise 5 quads into a (5, out_size, out_size) uint8 mask stack.'''
    stack = np.zeros((N_VERTEBRAE, out_size, out_size), np.uint8)
    sx, sy = out_size / float(W), out_size / float(H)
    for cid, quad in enumerate(polys):
        poly = np.asarray(quad, np.float32)
        if shrink != 1.0:
            ctr = poly.mean(0); poly = ctr + shrink * (poly - ctr)
        poly[:, 0] *= sx; poly[:, 1] *= sy
        cv2.fillPoly(stack[cid], [poly.astype(np.int32)], 1)
    return stack

# Build the list of valid (image, label) pairs ---------------------------------
def find_image(stem):
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
        p = os.path.join(IMG_DIR, stem + ext)
        if os.path.exists(p):
            return p
    return None

pairs = []
for lp in sorted(glob.glob(os.path.join(LBL_DIR, "*"))):
    stem = os.path.splitext(os.path.basename(lp))[0]
    ip = find_image(stem)
    if ip is None:
        continue
    if len(parse_polys(lp)) == N_VERTEBRAE:
        pairs.append((ip, lp))

print(f"Valid (image, label) pairs: {len(pairs)}")
assert len(pairs) > 0, "No valid pairs found - check BUU-LSPINE paths."


## Step 1 — Data Loading

We load the BUU-LSPINE AP X-rays (training/validation source) and register the DICOM test set.
**Visualization:** a grid of raw X-rays each paired with its pixel-intensity histogram.

In [ ]:
def load_gray_uint8(path):
    '''Load any X-ray (JPG/PNG or DICOM) as an 8-bit single-channel image (bright bone).'''
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        ds  = pydicom.dcmread(path)
        raw = ds.pixel_array.astype(np.float32)
        raw = raw * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            raw = raw.max() - raw                      # unify to 'bright bone'
        lo, hi = np.percentile(raw, 0.5), np.percentile(raw, 99.5)
        if hi <= lo:
            lo, hi = raw.min(), raw.max() + 1e-6
        return (np.clip((raw - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img

# ---- Visualization: raw samples + intensity histograms ----
def show_raw_with_histograms(sample_paths, title):
    n = len(sample_paths)
    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 6.4))
    for k, p in enumerate(sample_paths):
        g = load_gray_uint8(p)
        axes[0, k].imshow(g, cmap="gray")
        axes[0, k].set_title(os.path.basename(p)[:14], fontsize=9)
        axes[0, k].axis("off")
        axes[1, k].hist(g.ravel(), bins=64, color="steelblue", alpha=0.85)
        axes[1, k].set_xlabel("Pixel intensity"); axes[1, k].set_ylabel("Count")
        axes[1, k].set_title("Intensity histogram", fontsize=9)
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout(); plt.show()

sample_imgs = [p for p, _ in pairs[:4]]
show_raw_with_histograms(sample_imgs, "Step 1 — Raw BUU-LSPINE AP X-rays & intensity distributions")

test_dcm_files = sorted(glob.glob(os.path.join(TEST_DCM_DIR, "*.dcm")))
n_with_gt = sum(os.path.exists(os.path.join(TEST_MASK_DIR,
                os.path.splitext(os.path.basename(p))[0] + ".png")) for p in test_dcm_files)
print(f"DICOM test images registered: {len(test_dcm_files)} ({n_with_gt} with L4 ground-truth masks)")
if test_dcm_files:
    show_raw_with_histograms(test_dcm_files[:4], "Step 1 — Raw DICOM test X-rays & intensity distributions")

## Step 2 — Data Cleansing & Preprocessing

Pipeline: **median-blur artifact suppression → CLAHE contrast enhancement → resize → min–max
normalization** to $[0,1]$. CLAHE locally equalises trabecular contrast, which is critical for the
downstream GLCM texture analysis. **Visualization:** *Original Raw* vs *Preprocessed* side by side.

In [ ]:
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def preprocess_gray(gray_uint8, size=IMG_SIZE):
    '''Artifact removal + CLAHE + resize + min-max normalize -> float32 [0,1].'''
    g = cv2.medianBlur(gray_uint8, 3)                 # suppress salt-and-pepper artifacts
    g = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
    g = CLAHE.apply(g)                                # contrast-limited adaptive HE
    g = g.astype(np.float32)
    g = (g - g.min()) / (g.max() - g.min() + 1e-6)    # min-max normalize
    return g

# ---- Visualization: raw vs preprocessed ----
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k, p in enumerate(sample_imgs):
    raw  = load_gray_uint8(p)
    proc = preprocess_gray(raw)
    axes[0, k].imshow(raw, cmap="gray");  axes[0, k].set_title("Original Raw", fontsize=10); axes[0, k].axis("off")
    axes[1, k].imshow(proc, cmap="gray"); axes[1, k].set_title("Preprocessed (CLAHE+norm)", fontsize=10); axes[1, k].axis("off")
fig.suptitle("Step 2 — Original vs Preprocessed", fontsize=14, y=1.0)
plt.tight_layout(); plt.show()

## Step 3 — Train / Validation / Test Split & DataLoaders

The 400 BUU-LSPINE pairs are shuffled and split **80 / 20** into train / validation (both carry
ground-truth masks). The **DICOM `test/gather`** set is a separate inference-only cohort
(no masks). **Visualization:** sample distribution across the three sets.

In [ ]:
class SpineSegDataset(Dataset):
    '''Returns (image[1,H,W] float32, masks[5,H,W] float32) for U-Net training.'''
    def __init__(self, pairs, size=IMG_SIZE, augment=False):
        self.pairs = pairs; self.size = size; self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def _affine(self, x, y):
        '''Random small rotation + scale applied jointly to image and mask stack.'''
        S = self.size
        ang = random.uniform(-10, 10); sc = random.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((S / 2.0, S / 2.0), ang, sc)
        x = cv2.warpAffine(x, M, (S, S), flags=cv2.INTER_LINEAR, borderValue=0.0)
        y = np.stack([cv2.warpAffine(y[c], M, (S, S), flags=cv2.INTER_NEAREST, borderValue=0)
                      for c in range(y.shape[0])])
        return x, y

    def __getitem__(self, idx):
        ip, lp = self.pairs[idx]
        gray = load_gray_uint8(ip)
        H, W = gray.shape
        x = preprocess_gray(gray, self.size)                       # (S,S) float
        polys = parse_polys(lp)
        y = polys_to_maskstack(polys, H, W, self.size, shrink=1.0) # (5,S,S) uint8
        if self.augment:
            if random.random() < 0.5:                              # horizontal flip (AP-safe)
                x = np.ascontiguousarray(x[:, ::-1]); y = np.ascontiguousarray(y[:, :, ::-1])
            if random.random() < 0.5:                              # rotation + scale
                x, y = self._affine(x, y)
        x = torch.from_numpy(np.ascontiguousarray(x)).unsqueeze(0).float()
        y = torch.from_numpy(np.ascontiguousarray(y)).float()
        return x, y

# 80/20 split ------------------------------------------------------------------
shuffled = pairs[:]; random.Random(SEED).shuffle(shuffled)
n_val = int(len(shuffled) * VAL_RATIO)
val_pairs, train_pairs = shuffled[:n_val], shuffled[n_val:]

train_ds = SpineSegDataset(train_pairs, augment=True)
val_ds   = SpineSegDataset(val_pairs,   augment=False)

# Windows note: num_workers=0 avoids spawn issues inside notebooks
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

counts = {"Train": len(train_pairs), "Validation": len(val_pairs), "Test (DICOM)": len(test_dcm_files)}
print(counts)

# ---- Visualization: split distribution ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ["#4C72B0", "#55A868", "#C44E52"]
ax[0].bar(counts.keys(), counts.values(), color=colors)
for i, (k, v) in enumerate(counts.items()):
    ax[0].text(i, v + 1, str(v), ha="center", fontweight="bold")
ax[0].set_ylabel("Number of images"); ax[0].set_title("Sample distribution (bar)")
ax[1].pie(counts.values(), labels=counts.keys(), autopct="%1.1f%%", colors=colors, startangle=90)
ax[1].set_title("Sample distribution (pie)")
fig.suptitle("Step 3 — Dataset Split", fontsize=14); plt.tight_layout(); plt.show()

## Step 4 — Model Building (ResNet-34 U-Net, pretrained encoder)

A U-Net whose encoder is an **ImageNet-pretrained ResNet-34** (via `torchvision`). The grayscale
input is replicated to 3 channels to reuse the pretrained stem. Output: **5 logit channels**
(one binary mask per vertebra L1–L5); the **L4 channel (index 3)** plus a robust positional selector
drive the downstream BMD/texture/XAI analysis. A pretrained encoder converges much faster and
generalizes better across the JPG→DICOM domain gap than a from-scratch network — directly addressing
the previous version's frequent L4 misses. The deepest stage (`enc4`) is the Grad-CAM target.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

import torchvision

class ResNetUNet(nn.Module):
    '''U-Net with an ImageNet-pretrained ResNet-34 encoder (robust, fast-converging).'''
    def __init__(self, out_ch=N_CLASSES, pretrained=True):
        super().__init__()
        weights = None
        if pretrained:
            try:
                weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1
            except Exception:
                weights = None
        try:
            bb = torchvision.models.resnet34(weights=weights)
            if weights is None:
                print("[warn] Pretrained weights unavailable (offline?) - using random init.")
        except Exception:
            bb = torchvision.models.resnet34(weights=None)
            print("[warn] Could not fetch pretrained weights - using random init.")
        self.in_block = nn.Sequential(bb.conv1, bb.bn1, bb.relu)   # -> /2,  64
        self.pool = bb.maxpool
        self.enc1 = bb.layer1                                      # -> /4,  64
        self.enc2 = bb.layer2                                      # -> /8,  128
        self.enc3 = bb.layer3                                      # -> /16, 256
        self.enc4 = bb.layer4                                      # -> /32, 512  (Grad-CAM target)
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2); self.dec4 = DoubleConv(256 + 256, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2); self.dec3 = DoubleConv(128 + 128, 128)
        self.up2 = nn.ConvTranspose2d(128,  64, 2, stride=2); self.dec2 = DoubleConv(64 + 64,   64)
        self.up1 = nn.ConvTranspose2d(64,   64, 2, stride=2); self.dec1 = DoubleConv(64 + 64,   64)
        self.up0 = nn.ConvTranspose2d(64,   32, 2, stride=2); self.dec0 = DoubleConv(32,        32)
        self.head = nn.Conv2d(32, out_ch, 1)

    def forward(self, x):
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)              # grayscale -> 3ch for the pretrained stem
        x0 = self.in_block(x)                     # /2,  64
        x1 = self.enc1(self.pool(x0))             # /4,  64
        x2 = self.enc2(x1)                        # /8,  128
        x3 = self.enc3(x2)                        # /16, 256
        x4 = self.enc4(x3)                        # /32, 512
        d4 = self.dec4(torch.cat([self.up4(x4), x3], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), x2], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), x1], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), x0], 1))
        d0 = self.dec0(self.up0(d1))              # /1  (back to input resolution)
        return self.head(d0)

model = ResNetUNet(pretrained=True).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet34-UNet | trainable parameters: {n_params/1e6:.2f} M")

# Sanity check on a single batch
xb, yb = next(iter(train_loader))
with torch.no_grad():
    out = model(xb.to(DEVICE))
print("Input:", tuple(xb.shape), "-> Output logits:", tuple(out.shape), "(expected 5 mask channels)")

## Step 5 — Model Training (Hybrid Dice + BCE Loss)

**Objective:** $\mathcal{L} = \text{BCEWithLogits} + \text{SoftDice}$, averaged over the 5 mask
channels — BCE drives per-pixel correctness while Dice counters class imbalance (vertebrae occupy a
small fraction of the image). **Optimizer:** Adam. **Scheduler:** `ReduceLROnPlateau` on validation
loss. The best checkpoint (highest validation Dice) is saved to `best_model.pth`.
**Visualization:** train vs validation Loss and Dice over epochs.

In [ ]:
def dice_loss(logits, target, eps=1.0):
    probs = torch.sigmoid(logits)
    dims = (0, 2, 3)
    num = 2 * (probs * target).sum(dims) + eps
    den = (probs + target).sum(dims) + eps
    return 1.0 - (num / den).mean()

bce_fn = nn.BCEWithLogitsLoss()
def hybrid_loss(logits, target):
    return bce_fn(logits, target) + dice_loss(logits, target)

@torch.no_grad()
def dice_coef(logits, target, thr=0.5, eps=1.0):
    probs = (torch.sigmoid(logits) > thr).float()
    dims = (0, 2, 3)
    num = 2 * (probs * target).sum(dims) + eps
    den = (probs + target).sum(dims) + eps
    return (num / den).mean().item()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": []}
best_val_dice = -1.0

def run_epoch(loader, train=True):
    model.train(train)
    tot_loss = tot_dice = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(xb)
            loss = hybrid_loss(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot_loss += loss.item() * xb.size(0)
        tot_dice += dice_coef(logits, yb) * xb.size(0)
    n = len(loader.dataset)
    return tot_loss / n, tot_dice / n

print(f"Training on {DEVICE} for {EPOCHS} epochs ...")
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_dice = run_epoch(train_loader, train=True)
    va_loss, va_dice = run_epoch(val_loader,   train=False)
    scheduler.step(va_loss)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_dice"].append(tr_dice); history["val_dice"].append(va_dice)
    flag = ""
    if va_dice > best_val_dice:
        best_val_dice = va_dice
        torch.save(model.state_dict(), BEST_MODEL_PATH); flag = "  <-- best saved"
    print(f"Epoch {ep:02d}/{EPOCHS} | train loss {tr_loss:.4f} dice {tr_dice:.4f} "
          f"| val loss {va_loss:.4f} dice {va_dice:.4f}{flag}")
print(f"Done in {(time.time()-t0)/60:.1f} min | best val Dice = {best_val_dice:.4f}")

# ---- Visualization: learning curves ----
ep_axis = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(ep_axis, history["train_loss"], "-o", label="Train loss")
ax[0].plot(ep_axis, history["val_loss"],   "-s", label="Validation loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Hybrid loss (BCE+Dice)")
ax[0].set_title("Training vs Validation Loss"); ax[0].legend()
ax[1].plot(ep_axis, history["train_dice"], "-o", label="Train Dice")
ax[1].plot(ep_axis, history["val_dice"],   "-s", label="Validation Dice")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Dice coefficient")
ax[1].set_title("Training vs Validation Dice"); ax[1].legend()
fig.suptitle("Step 5 — Learning Curves", fontsize=14); plt.tight_layout(); plt.show()

### BMD & GLCM feature extractors

Reusable functions that compute the **BMD intensity index** and the **GLCM texture descriptors**
from a normalized image and a binary L4 mask. These are used both in Evaluation (Step 6) and in the
inference function (Step 7).

In [ ]:
def shrink_mask(mask, shrink=ROI_SHRINK):
    '''Erode mask toward its centroid to sample central cancellous bone.'''
    if shrink >= 1.0:
        return mask
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return mask
    cx, cy = xs.mean(), ys.mean()
    out = np.zeros_like(mask)
    nx = (cx + shrink * (xs - cx)).astype(int)
    ny = (cy + shrink * (ys - cy)).astype(int)
    nx = np.clip(nx, 0, mask.shape[1]-1); ny = np.clip(ny, 0, mask.shape[0]-1)
    out[ny, nx] = 1
    return cv2.morphologyEx(out, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))

def compute_bmd(img_norm, mask):
    '''Area-based BMD proxy: intensity statistics inside the L4 ROI (img_norm in [0,1]).'''
    vals = img_norm[mask > 0]
    if vals.size == 0:
        return None
    return {"area_px": int(vals.size),
            "bmd_mean":   float(vals.mean()),
            "bmd_median": float(np.median(vals)),
            "bmd_std":    float(vals.std()),
            "bmd_p10":    float(np.percentile(vals, 10)),
            "bmd_p90":    float(np.percentile(vals, 90))}

def compute_glcm(img_norm, mask, levels=GLCM_LEVELS):
    '''GLCM texture descriptors over the L4 ROI bounding box (background masked out).'''
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    roi  = (img_norm[y0:y1, x0:x1] * (levels - 1)).astype(np.uint8)
    mroi = mask[y0:y1, x0:x1]
    roi[mroi == 0] = 0
    glcm = graycomatrix(roi, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=levels, symmetric=True, normed=True)
    return {"glcm_contrast":    float(graycoprops(glcm, "contrast").mean()),
            "glcm_correlation": float(graycoprops(glcm, "correlation").mean()),
            "glcm_energy":      float(graycoprops(glcm, "energy").mean()),
            "glcm_homogeneity": float(graycoprops(glcm, "homogeneity").mean())}

@torch.no_grad()
def predict_masks(img_norm):
    '''Run the model on a normalized [0,1] image -> per-class probability maps (5,S,S).'''
    model.eval()
    x = torch.from_numpy(img_norm).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    probs = torch.sigmoid(model(x))[0].cpu().numpy()
    return probs

def _largest_cc(binary):
    binary = binary.astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n <= 1:
        return binary
    biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return (lab == biggest).astype(np.uint8)

def l4_binary_mask(probs, thr=0.5):
    '''Robust L4 selection.

    Vertebrae look near-identical, so trusting the single L4 channel is fragile. Instead we
    build the combined vertebra foreground, order the blobs top->bottom and take the 4th (= L4),
    cross-checked against the L4 channel. Multi-stage fallbacks guarantee a non-empty mask
    (mirrors the YOLO prototype's '4th-from-top' resilience).
    '''
    C, H, W = probs.shape
    min_area = max(20, int(0.0006 * H * W))

    # 1) Combined vertebra foreground -> ordered components (top -> bottom)
    fg = (probs.max(0) > thr).astype(np.uint8)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, stats, cent = cv2.connectedComponentsWithStats(fg, connectivity=8)
    comps = sorted([i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area],
                   key=lambda i: cent[i][1])

    pos_mask = None
    if comps:
        if len(comps) >= 5:
            pos_mask = (lab == comps[3]).astype(np.uint8)          # exact 4th-from-top = L4
        else:
            j = int(round(0.75 * (len(comps) - 1)))                # L4 sits ~75% down the lumbar run
            pos_mask = (lab == comps[min(j, len(comps) - 1)]).astype(np.uint8)

    # 2) Direct L4-channel evidence, cross-checked with the positional candidate
    direct = (probs[L4_IDX] > thr).astype(np.uint8)
    if direct.sum() >= min_area:
        d = _largest_cc(direct)
        if pos_mask is not None and (d & pos_mask).sum() >= 0.15 * d.sum():
            return pos_mask                                        # agreement -> use clean blob
        if pos_mask is None:
            return d
        return pos_mask                                            # disagreement -> trust position

    # 3) Fallbacks: positional candidate, else relaxed-threshold L4 channel
    if pos_mask is not None:
        return pos_mask
    return _largest_cc((probs[L4_IDX] > 0.3 * thr).astype(np.uint8))

def load_test_gt_l4(dcm_path, size=IMG_SIZE):
    '''Load the binary L4 ground-truth mask (PNG) matching a test DICOM, resized to size.'''
    stem = os.path.splitext(os.path.basename(dcm_path))[0]
    p = os.path.join(TEST_MASK_DIR, stem + ".png")
    if not os.path.exists(p):
        return None
    m = cv2.imread(p)                                  # BGR; foreground is non-black
    fg = (m.sum(axis=2) > 10).astype(np.uint8)
    return cv2.resize(fg, (size, size), interpolation=cv2.INTER_NEAREST)
print("BMD / GLCM / prediction / GT-mask helpers ready.")

## Step 6 — Evaluation (on the DICOM Test Set)

We load the **best checkpoint** and evaluate on the held-out **DICOM test cohort**
(`dataset-dcm/test/gather`), which ships with **L4 ground-truth masks**
(`gather_mask/SegmentationClass/*.png`). This is a genuine cross-domain test: the model is trained on
BUU-LSPINE JPG AP X-rays and evaluated on independent DICOM studies.

Reported per the requirement: **Dice, IoU, Precision, Recall** for the L4 mask, a **pixel confusion
matrix**, predicted-mask **overlays** (with GT contour), and a **Bland–Altman + correlation** plot
comparing the BMD measured inside the *predicted* L4 mask against the BMD inside the *ground-truth*
L4 mask.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
model.eval()

def seg_scores(pred, gt, eps=1e-6):
    pred = pred.astype(bool); gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    dice = 2*tp / (2*tp + fp + fn + eps)
    iou  = tp / (tp + fp + fn + eps)
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    return dice, iou, prec, rec, tp, fp, fn

# ---- Aggregate L4 metrics + density agreement over the DICOM test set ----
eval_items = [p for p in test_dcm_files
              if os.path.exists(os.path.join(TEST_MASK_DIR,
                 os.path.splitext(os.path.basename(p))[0] + ".png"))]
print(f"Evaluating on {len(eval_items)} test images with L4 ground truth.")

rows, bmd_pred, bmd_gt = [], [], []
cm_total = np.zeros((2, 2), dtype=np.int64)   # pixel confusion for L4 (bg vs L4)
for dp in eval_items:
    gray = load_gray_uint8(dp)
    img  = preprocess_gray(gray)
    probs = predict_masks(img)
    pred_m = l4_binary_mask(probs)
    gt_m   = load_test_gt_l4(dp)
    if gt_m is None:
        continue

    d, i, p, r, tp, fp, fn = seg_scores(pred_m, gt_m)
    tn = IMG_SIZE*IMG_SIZE - (tp + fp + fn)
    cm_total += np.array([[tn, fp], [fn, tp]], dtype=np.int64)
    rows.append({"file": os.path.basename(dp), "dice": d, "iou": i, "precision": p, "recall": r})

    bp = compute_bmd(img, shrink_mask(pred_m))
    bg = compute_bmd(img, shrink_mask(gt_m))
    if bp and bg:
        bmd_pred.append(bp["bmd_mean"]); bmd_gt.append(bg["bmd_mean"])

dfm = pd.DataFrame(rows)
print("\n=== L4 segmentation metrics on DICOM test set (mean +/- std) ===")
for c in ["dice", "iou", "precision", "recall"]:
    print(f"  {c.capitalize():10s}: {dfm[c].mean():.4f} +/- {dfm[c].std():.4f}")
dfm.to_csv(os.path.join(OUT_DIR, "L4_test_segmetrics.csv"), index=False)

In [ ]:
# ---- Visualization 1: predicted (green) vs ground-truth (yellow contour) overlays ----
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for k, dp in enumerate(eval_items[:8]):
    gray = load_gray_uint8(dp); img = preprocess_gray(gray)
    pred_m = l4_binary_mask(predict_masks(img))
    gt_m   = load_test_gt_l4(dp)
    rgb = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    rgb[pred_m > 0] = (0.5*rgb[pred_m > 0] + 0.5*np.array([0, 255, 0])).astype(np.uint8)
    if gt_m is not None:
        cnts, _ = cv2.findContours(gt_m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(rgb, cnts, -1, (255, 255, 0), 2)
    ax = axes[k//4, k % 4]
    ax.imshow(rgb); ax.set_title(os.path.basename(dp)[-12:], fontsize=8); ax.axis("off")
fig.suptitle("Step 6 — Predicted L4 mask (green) vs ground truth (yellow) on DICOM test X-rays", fontsize=13)
plt.tight_layout(); plt.show()

# ---- Visualization 2: pixel confusion matrix (L4 vs background) ----
fig, ax = plt.subplots(figsize=(5.2, 4.4))
cm_norm = cm_total / cm_total.sum()
sns.heatmap(cm_norm, annot=True, fmt=".4f", cmap="Blues",
            xticklabels=["Pred bg", "Pred L4"], yticklabels=["True bg", "True L4"], ax=ax)
ax.set_title("Step 6 — Pixel Confusion Matrix (L4 vs background, normalized)")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Visualization 3: Bland-Altman + correlation of BMD (predicted vs GT mask) ----
bmd_pred = np.array(bmd_pred); bmd_gt = np.array(bmd_gt)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Correlation scatter
ax[0].scatter(bmd_gt, bmd_pred, alpha=0.6, color="#4C72B0")
lims = [min(bmd_gt.min(), bmd_pred.min()), max(bmd_gt.max(), bmd_pred.max())]
ax[0].plot(lims, lims, "r--", label="y = x")
if len(bmd_gt) > 1:
    r = np.corrcoef(bmd_gt, bmd_pred)[0, 1]
    ax[0].set_title(f"BMD agreement (Pearson r = {r:.3f})")
ax[0].set_xlabel("BMD index (ground-truth mask)"); ax[0].set_ylabel("BMD index (predicted mask)")
ax[0].legend()

# Bland-Altman
mean_b = (bmd_pred + bmd_gt) / 2.0
diff_b = bmd_pred - bmd_gt
md_, sd_ = diff_b.mean(), diff_b.std()
ax[1].scatter(mean_b, diff_b, alpha=0.6, color="#55A868")
ax[1].axhline(md_, color="k", label=f"mean diff = {md_:.4f}")
ax[1].axhline(md_ + 1.96*sd_, color="r", ls="--", label="+1.96 SD")
ax[1].axhline(md_ - 1.96*sd_, color="r", ls="--", label="-1.96 SD")
ax[1].set_xlabel("Mean BMD index"); ax[1].set_ylabel("Predicted - Ground-truth")
ax[1].set_title("Bland-Altman (BMD agreement)"); ax[1].legend()
fig.suptitle("Step 6 — BMD agreement on DICOM test set: predicted vs ground-truth L4 mask", fontsize=13)
plt.tight_layout(); plt.show()

## Step 7 — Pre-trained Model & Modular Inference

The best weights are already saved at `best_model.pth`. Below is a **single modular inference
function** that accepts any raw X-ray path (**JPG/PNG or DICOM**), runs segmentation, and returns +
plots the **L4 BMD and microarchitecture (GLCM)** metrics. We then run it on the DICOM test cohort.

In [ ]:
def infer_l4(path, model=model, show=True, save_csv=False):
    '''End-to-end inference: raw X-ray path -> segmentation plot + L4 BMD/GLCM metrics dict.'''
    gray = load_gray_uint8(path)
    img  = preprocess_gray(gray)
    probs = predict_masks(img)
    pred_m = l4_binary_mask(probs)
    roi_m  = shrink_mask(pred_m)

    bmd  = compute_bmd(img, roi_m)
    glcm = compute_glcm(img, roi_m)
    metrics = {"file": os.path.basename(path)}
    if bmd:  metrics.update(bmd)
    if glcm: metrics.update(glcm)

    if show:
        rgb = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
        overlay = rgb.copy()
        overlay[pred_m > 0] = (0.5*overlay[pred_m > 0] + 0.5*np.array([0, 255, 0])).astype(np.uint8)
        overlay[roi_m   > 0] = (0.5*overlay[roi_m   > 0] + 0.5*np.array([255, 0, 0])).astype(np.uint8)
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(img, cmap="gray"); ax[0].set_title("Preprocessed input"); ax[0].axis("off")
        ax[1].imshow(probs[L4_IDX], cmap="magma"); ax[1].set_title("L4 probability map"); ax[1].axis("off")
        ax[2].imshow(overlay); ax[2].axis("off")
        ax[2].set_title("L4 segmentation (green=mask, red=cancellous ROI)")
        if bmd and glcm:
            txt = (f"BMD mean={bmd['bmd_mean']:.3f}  median={bmd['bmd_median']:.3f}\n"
                   f"GLCM contrast={glcm['glcm_contrast']:.2f}  energy={glcm['glcm_energy']:.3f}\n"
                   f"correlation={glcm['glcm_correlation']:.3f}  homog.={glcm['glcm_homogeneity']:.3f}")
            ax[2].text(0.02, 0.98, txt, transform=ax[2].transAxes, va="top", fontsize=9,
                       color="white", bbox=dict(facecolor="black", alpha=0.6))
        fig.suptitle(f"Inference — {os.path.basename(path)[:24]}", fontsize=13)
        plt.tight_layout(); plt.show()
    return metrics

# ---- Run inference on the DICOM test cohort & build a results table ----
results = []
for p in test_dcm_files[:6]:          # plot first 6; loop over all for the CSV below
    results.append(infer_l4(p, show=True))
for p in test_dcm_files[6:]:
    results.append(infer_l4(p, show=False))

if results:
    res_df = pd.DataFrame(results)
    res_df.to_csv(os.path.join(OUT_DIR, "L4_test_metrics.csv"), index=False)
    print(f"\nSaved metrics for {len(res_df)} test images -> {os.path.join(OUT_DIR, 'L4_test_metrics.csv')}")
    display(res_df.head(10))

## 3. Explainable AI (XAI) — Grad-CAM on the U-Net Bottleneck

To make the segmentation clinically auditable we apply **Grad-CAM** to the deepest convolutional
stage of the encoder (the ResNet **`enc4`** block). The target scalar is the mean L4-channel logit over the predicted
L4 region; its gradient w.r.t. the bottleneck activations is global-average-pooled into channel
weights, combined, ReLU-rectified, up-sampled, and blended over the X-ray. Bright regions show the
structures (endplates, cortical edges, trabecular density) that most influenced the L4 decision.

In [ ]:
class GradCAM:
    '''Grad-CAM for the segmentation U-Net, hooked on the bottleneck block.'''
    def __init__(self, model, target_layer):
        self.model = model; self.acts = None; self.grads = None
        target_layer.register_forward_hook(self._fwd)
        target_layer.register_full_backward_hook(self._bwd)
    def _fwd(self, m, i, o): self.acts = o.detach()
    def _bwd(self, m, gi, go): self.grads = go[0].detach()

    def __call__(self, img_norm, class_idx=L4_IDX):
        self.model.eval()
        x = torch.from_numpy(img_norm).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits = self.model(x)
        with torch.no_grad():
            region = (torch.sigmoid(logits[0, class_idx]) > 0.5).float()
        if region.sum() < 1:
            region = torch.ones_like(region)
        score = (logits[0, class_idx] * region).sum() / (region.sum() + 1e-6)
        self.model.zero_grad(); score.backward()
        weights = self.grads.mean(dim=(2, 3), keepdim=True)            # GAP over spatial dims
        cam = F.relu((weights * self.acts).sum(dim=1, keepdim=True))   # (1,1,h,w)
        cam = F.interpolate(cam, size=img_norm.shape, mode="bilinear", align_corners=False)
        cam = cam[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-6)
        return cam

gradcam = GradCAM(model, model.enc4)

# ---- Visualization: Grad-CAM blended overlay on validation samples ----
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for k, (ip, lp) in enumerate(val_pairs[:4]):
    gray = load_gray_uint8(ip); img = preprocess_gray(gray)
    cam  = gradcam(img)
    pred_m = l4_binary_mask(predict_masks(img))
    rgb = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    heat = cv2.applyColorMap((cam*255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    blend = (0.55*rgb + 0.45*heat).astype(np.uint8)
    # outline predicted L4 for context
    cnts, _ = cv2.findContours(pred_m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(blend, cnts, -1, (0, 255, 0), 2)
    axes[0, k].imshow(rgb);   axes[0, k].set_title(os.path.basename(ip)[:12], fontsize=9); axes[0, k].axis("off")
    axes[1, k].imshow(blend); axes[1, k].set_title("Grad-CAM (L4) + mask outline", fontsize=9); axes[1, k].axis("off")
axes[0, 0].set_ylabel("Input", fontsize=11)
fig.suptitle("XAI — Grad-CAM attribution for the L4 segmentation decision", fontsize=14)
plt.tight_layout(); plt.show()

## Summary & Clinical Notes

- **Segmentation:** a ResNet-34 U-Net (ImageNet-pretrained encoder, 5 channels L1–L5) trained with a
  hybrid **Dice + BCE** loss; a **robust positional selector** turns the predictions into the final
  L4 mask without dropped detections.
- **BMD index:** normalized mean intensity inside the predicted L4 ROI (central cancellous bone via
  `ROI_SHRINK`). This is a **relative** radiographic proxy — for absolute BMD (g/cm²) an in-image
  aluminium step-wedge or DXA/QCT calibration is required.
- **Microarchitecture:** rotation-averaged GLCM (Contrast, Correlation, Energy/ASM, Homogeneity)
  characterising trabecular texture.
- **Trust:** Grad-CAM confirms the network attends to the L4 vertebral body / endplates rather than
  background artifacts.
- **Reproducibility:** best weights at `best_model.pth`; the modular `infer_l4()` runs on any
  JPG/PNG/DICOM X-ray.

**To improve accuracy:** add an in-image calibration phantom, soft-tissue normalization, manual
mask refinement for a subset, and longer training (`EPOCHS`) with stronger augmentation.